# Coverage by Road Analysis

This notebook downloads the drivable street network for New York City using `osmnx`, buffers the street segments to create polygons, and then polyfills these polygons with H3 hexagons at resolution 12 using `h3-py`.

In [ ]:
# Install dependencies if not already installed
# !pip install osmnx h3 geopandas shapely matplotlib

In [ ]:
import osmnx as ox
import h3
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import shape
import matplotlib.pyplot as plt
import requests

# Configuration
# PLACE = "New York City, New York, USA"
SOCRATA_URL = "https://data.cityofnewyork.us/resource/inkn-q76z.geojson"
H3_RES = 12
# BUFFER_DIST_METERS is removed as we now use 'streetwidth' column
DEFAULT_STREET_WIDTH_FT = 30 # Fallback if streetwidth is missing/invalid

print(f"Using H3 library version: {h3.__version__}")

## 1. Download Street Network

We use `osmnx` to fetch the drivable street network for the specified place.

In [ ]:
print("Downloading street network from NYC Open Data...")

def fetch_socrata_geojson(url, limit=50000):
    offset = 0
    all_features = []
    
    while True:
        print(f"Fetching records with offset {offset}...")
        params = {
            "$limit": limit,
            "$offset": offset,
            "$order": ":id" # Sort by ID to ensure stable pagination
        }
        
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error fetching data: {e}")
            break
            
        features = data.get('features', [])
        if not features:
            break
            
        all_features.extend(features)
        
        if len(features) < limit:
            break
            
        offset += limit
        
    return {"type": "FeatureCollection", "features": all_features}

geojson_data = fetch_socrata_geojson(SOCRATA_URL)
print(f"Download complete. Fetched {len(geojson_data['features'])} features.")

## 2. Process Geometries

Convert the graph to a GeoDataFrame and buffer the edges to create polygons suitable for H3 polyfilling.

In [ ]:
# Convert to GeoDataFrame
gdf_edges = gpd.GeoDataFrame.from_features(geojson_data)
# Ensure CRS is set (GeoJSON is typically EPSG:4326)
gdf_edges.set_crs(epsg=4326, inplace=True)

# Filter out non-roads based on 'rw_type'
# Keep only rw_type 1 (Street), 2 (Highway), and 9 (Depressed/Ramp) as requested.
# Filter out elevated roadways (bridges, viaducts) which typically have level codes > 13.
# 13 is Ground, 9 is Sub-surface/Depressed. 17, 21+ are Elevated.

# First, ensure columns are numeric
gdf_edges['rw_type'] = pd.to_numeric(gdf_edges['rw_type'], errors='coerce')
gdf_edges['from_level_code'] = pd.to_numeric(gdf_edges['from_level_code'], errors='coerce').fillna(13)
gdf_edges['to_level_code'] = pd.to_numeric(gdf_edges['to_level_code'], errors='coerce').fillna(13)

# 1. Keep specific rw_types (1, 2, 9)
# 3 (Bridges) and 4 (Tunnels) are excluded by this list.
gdf_edges = gdf_edges[gdf_edges['rw_type'].isin([1, 2, 9])]

# 2. Exclude elevated roadways (Level > 13)
# We keep Ground (13) and Depressed (<13).
is_elevated = (gdf_edges['from_level_code'] > 13) | (gdf_edges['to_level_code'] > 13)
gdf_edges = gdf_edges[~is_elevated]

print(f"Number of street segments (edges) after filtering types (1,2,9) and removing elevated: {len(gdf_edges)}")

# Process 'streetwidth' column
# Convert to numeric, handling errors
gdf_edges['streetwidth'] = pd.to_numeric(gdf_edges['streetwidth'], errors='coerce')

# Fill NaNs with default width (e.g. 30 ft)
gdf_edges['streetwidth'] = gdf_edges['streetwidth'].fillna(DEFAULT_STREET_WIDTH_FT)

# Convert feet to meters (1 ft = 0.3048 m)
# The buffer radius is half the street width
gdf_edges['buffer_radius_meters'] = (gdf_edges['streetwidth'] * 0.3048) / 2.0

print(f"Average buffer radius (m): {gdf_edges['buffer_radius_meters'].mean():.2f}")

# Project to a projected CRS (UTM) to buffer in meters
# estimate_utm_crs() finds the best local projection
utm_crs = gdf_edges.estimate_utm_crs()
gdf_edges_proj = gdf_edges.to_crs(utm_crs)

# Buffer the LineStrings to create Polygons using the variable radius
# We can use the buffer method on the geometry column but with a distance argument
# However, geopandas buffer accepts a single distance or an array-like if it matches index
# but for safety with different versions, let's apply or use the vectorized version if supported

# In newer GeoPandas/Shapely, we can pass the array directly if indices align
try:
    gdf_edges_proj['geometry'] = gdf_edges_proj.geometry.buffer(gdf_edges_proj['buffer_radius_meters'])
except:
    # Fallback for older versions or if direct array passing fails
    gdf_edges_proj['geometry'] = gdf_edges_proj.apply(
        lambda row: row.geometry.buffer(row['buffer_radius_meters']), axis=1
    )

# Project back to EPSG:4326 (Latitude/Longitude) for H3
gdf_edges_buffered = gdf_edges_proj.to_crs(epsg=4326)

# Filter out any empty geometries if they exist
gdf_edges_buffered = gdf_edges_buffered[~gdf_edges_buffered.is_empty]

print("Buffering complete. Example geometry:")
print(gdf_edges_buffered.geometry.iloc[0])


In [ ]:
gdf_edges.columns

## 3. Polyfill with H3

We use `h3.geo_to_cells` to fill the street polygons with hexagons.

In [ ]:
def get_hexagons(geometry, res):
    """
    Polyfills a shapely geometry with H3 hexagons at the given resolution.
    Uses h3.geo_to_cells which supports __geo_interface__.
    """
    try:
        # h3.geo_to_cells takes a GeoJSON-like object or object with __geo_interface__
        # It returns a set/list of cell indices
        return h3.geo_to_cells(geometry, res=res)
    except Exception as e:
        # Handle cases where geometry might be invalid or empty
        return set()

print(f"Polyfilling {len(gdf_edges_buffered)} segments with H3 resolution {H3_RES}...")

# Apply the polyfill function
gdf_edges_buffered['h3_cells'] = gdf_edges_buffered.geometry.apply(lambda x: get_hexagons(x, H3_RES))

# Flatten the list of sets to get all unique cells covering the network
all_network_cells = set()
for cell_set in gdf_edges_buffered['h3_cells']:
    all_network_cells.update(cell_set)

print(f"Total unique H3 cells covering the network: {len(all_network_cells)}")

## 4. Visualization (Optional)

Visualize the coverage by converting a subset of cells back to geometries.

In [ ]:
# Convert a sample of cells back to polygons for visualization
sample_size = min(1000, len(all_network_cells))
sample_cells = list(all_network_cells)[:sample_size]
sample_cells = None

if sample_cells:
    # Use H3 helper to create a MultiPolygon from cells
    # h3.cells_to_h3shape returns a LatLngMultiPoly object with __geo_interface__
    h3_shape = h3.cells_to_h3shape(sample_cells)
    
    # Convert to Shapely geometry
    shapely_geom = shape(h3_shape.__geo_interface__)
    
    gdf_hex = gpd.GeoDataFrame({'geometry': [shapely_geom]}, crs="EPSG:4326")
    # Explode mutipolygon to individual polygons for easier plotting if needed, 
    # or just plot the multipolygon
    gdf_hex = gdf_hex.explode(index_parts=False)

    fig, ax = plt.subplots(figsize=(10, 10))
    # Plot intersections (nodes) - REMOVED as we don't have nodes anymore
    # gdf_nodes.plot(ax=ax, markersize=1, color='red', alpha=0.5, label='Intersections')
    # Plot hexagons
    gdf_hex.plot(ax=ax, alpha=0.6, edgecolor='blue', facecolor='none', label='H3 Cells (Sample)')
    ax.set_title(f"Sample of H3 Coverage (Res {H3_RES})")
    plt.legend()
    plt.show()
else:
    print("No cells to visualize.")

## 5. Load Dashcam Metadata & Analyze Coverage

Load the Nexar dashcam dataset metadata and compare against the street network coverage.

In [ ]:
# Load metadata
md_path = '../../data/processed/md.csv'
print(f"Loading dashcam metadata from {md_path}...")
df_md = pd.read_csv(md_path)

# Filter out invalid coordinates if any
df_md = df_md.dropna(subset=['gps_info.latitude', 'gps_info.longitude'])

print(f"Loaded {len(df_md)} dashcam records.")

In [ ]:
# Compute H3 index for each image
def get_h3_index(row, res):
    return h3.latlng_to_cell(row['gps_info.latitude'], row['gps_info.longitude'], res)

print(f"Computing H3 indices (Res {H3_RES}) for dashcam images...")
df_md['h3_cell'] = df_md.apply(lambda row: get_h3_index(row, H3_RES), axis=1)

# Count images per cell
cell_counts = df_md['h3_cell'].value_counts()
print(f"Found {len(cell_counts)} unique cells with images.")

In [ ]:
# Compare with Street Network Coverage
# Create a DataFrame of all network cells
df_network = pd.DataFrame({'h3_cell': list(all_network_cells)})

# Merge with image counts
df_coverage = df_network.merge(
    cell_counts.rename('image_count'), 
    left_on='h3_cell', 
    right_index=True, 
    how='left'
)

# Fill missing counts with 0
df_coverage['image_count'] = df_coverage['image_count'].fillna(0).astype(int)

# Create binary column
df_coverage['has_images'] = df_coverage['image_count'] > 0

# Statistics
total_cells = len(df_coverage)
cells_with_images = df_coverage['has_images'].sum()
cells_without_images = total_cells - cells_with_images
percent_covered = (cells_with_images / total_cells) * 100

print("--- Coverage Statistics ---")
print(f"Total Street Network Cells: {total_cells}")
print(f"Cells with Images: {cells_with_images} ({percent_covered:.2f}%)")
print(f"Cells without Images: {cells_without_images} ({100 - percent_covered:.2f}%)")
print(f"Max images in a single cell: {df_coverage['image_count'].max()}")

In [ ]:
# Plot Distribution
plt.figure(figsize=(10, 6))
plt.hist(df_coverage[df_coverage['image_count'] > 0]['image_count'], bins=50, log=True)
plt.title('Distribution of Image Counts per Cell (Log Scale)')
plt.xlabel('Number of Images')
plt.ylabel('Number of Cells (Log Scale)')
plt.show()

In [ ]:
# fraction of cells with 0 images 


## 6. Map Visualization of Coverage

Visualize the distribution on a map.

In [ ]:
# Prepare GeoDataFrame for visualization
# We'll map ALL network cells, coloring by whether they have images or the count

# Define the upscale resolution for visualization
# H3 resolution 12 is small. Resolution 9 is larger.
VIS_RES = 11

print(f"Upscaling cells to resolution {VIS_RES} for better visibility on map...")

# Function to get parent cell at lower resolution
def get_parent_cell(h3_index, res):
    return h3.cell_to_parent(h3_index, res)

# Add parent cell column
df_coverage['h3_cell_vis'] = df_coverage['h3_cell'].apply(lambda x: get_parent_cell(x, VIS_RES))

# Aggregate counts by parent cell
# We want to sum image counts and keep track if ANY child cell had images
df_vis = df_coverage.groupby('h3_cell_vis').agg({
    'image_count': 'sum',
    'has_images': 'max' # will be 1 (True) if any child has images
}).reset_index()

print(f"Aggregated to {len(df_vis)} parent cells at resolution {VIS_RES}.")

# Convert to Geometry
def cell_to_geom(h3_index):
    try:
        boundary = h3.cell_to_boundary(h3_index)
        boundary_lonlat = [(p[1], p[0]) for p in boundary]
        return shape({'type': 'Polygon', 'coordinates': [boundary_lonlat]})
    except:
        return None

df_vis['geometry'] = df_vis['h3_cell_vis'].apply(cell_to_geom)
gdf_vis = gpd.GeoDataFrame(df_vis, geometry='geometry', crs='EPSG:4326')

# Plotting
fig, ax = plt.subplots(figsize=(15, 15))

# Plot ALL cells
# We use a log scale or quantiles for coloring
# Cells with 0 images will be plotted too
gdf_vis.plot(column='image_count', ax=ax, legend=True, 
             cmap='viridis', scheme='quantiles', k=5, 
             linewidth=0.1, edgecolor='white',
             legend_kwds={'loc': 'lower right', 'title': 'Image Count (Aggregated)'})

ax.set_title(f"Dashcam Coverage Map (Aggregated to Res {VIS_RES})")
plt.axis('off')
plt.show()

## 7. Multi-Resolution Coverage Analysis

Calculate coverage statistics for H3 resolutions 10, 11, and 12.

In [ ]:
def analyze_coverage_at_resolution(resolution, network_gdf, dashcam_df):
    print(f"\n--- Analyzing Coverage at Resolution {resolution} ---")
    
    # 1. Get Street Network Cells at this resolution
    # We need to re-polyfill the street network at the target resolution
    # Or, if we are sure res 12 covers everything fine, we can just aggregate res 12 up to target res
    # But polyfilling again is more rigorous to ensure edge cases are handled correctly.
    # However, polyfilling 120k polygons can take time. 
    # Let's reuse the buffer geometries.
    
    print(f"Polyfilling street network at res {resolution}...")
    # Use a vectorized approach or list comprehension if faster
    # Re-using the 'get_hexagons' function defined earlier
    network_cells = set()
    # We'll iterate and update set
    # Optimizing: can we use h3.cell_to_parent on the existing res 12 cells?
    # Technically yes, if res 12 covers the road fully, the parents of those cells cover the road too.
    # This is much faster.
    
    if resolution <= H3_RES: # H3_RES is 12
        print("Aggregating from Res 12 network cells (faster)...")
        for cell in all_network_cells:
            network_cells.add(h3.cell_to_parent(cell, resolution))
    else:
        # Fallback if we ever want higher res (unlikely here)
        print("Polyfilling from geometry...")
        for geom in network_gdf.geometry:
             network_cells.update(h3.geo_to_cells(geom, res=resolution))
            
    print(f"Total Street Network Cells (Res {resolution}): {len(network_cells)}")
    
    # 2. Get Dashcam Cells at this resolution
    # We can calculate these on the fly or aggregate from existing 'h3_cell' (res 12) column
    print("Computing dashcam cells...")
    dashcam_cells = dashcam_df['h3_cell'].apply(lambda x: h3.cell_to_parent(x, resolution))
    dashcam_counts = dashcam_cells.value_counts()
    
    # 3. Merge and Statistics
    df_res = pd.DataFrame({'h3_cell': list(network_cells)})
    df_res = df_res.merge(
        dashcam_counts.rename('image_count'),
        left_on='h3_cell',
        right_index=True,
        how='left'
    )
    
    df_res['image_count'] = df_res['image_count'].fillna(0).astype(int)
    df_res['has_images'] = df_res['image_count'] > 0
    
    total = len(df_res)
    covered = df_res['has_images'].sum()
    not_covered = total - covered
    pct_covered = (covered / total) * 100
    
    print(f"Total Cells: {total}")
    print(f"Cells with Images: {covered} ({pct_covered:.2f}%)")
    print(f"Cells without Images: {not_covered} ({100 - pct_covered:.2f}%)")
    
    return df_res

# Run analysis for 10, 11, 12
resolutions = [10, 11, 12]
coverage_stats = {}

for res in resolutions:
    coverage_stats[res] = analyze_coverage_at_resolution(res, gdf_edges_buffered, df_md)

## 8. Minimaps for Each Resolution

Generate smaller maps for H3 resolutions 10, 11, and 12 to visualize coverage density at different granularities.

In [ ]:
def plot_minimap(ax, coverage_df, resolution):
    # Upscale to a common resolution for visualization if needed, 
    # OR just plot the cells at their native resolution.
    # Plotting native resolution 12 might be hard to see on a small map, 
    # but let's try native first as requested to show "validated h3 coverage resolution".
    
    # However, plotting 400k polys for res 12 is heavy. 
    # We will aggregate to a visible resolution (e.g. Res 9 or 10) for the plot ITSELF, 
    # but based on the data from the specific resolution analysis.
    # actually, let's just plot the native cells but maybe downsample if too huge? 
    # No, let's stick to the aggregation approach for visibility, OR just plot native.
    # Given "validated h3 coverage resolution", user likely wants to see the actual grid.
    # But drawing 400k patches is very slow. 
    # Let's use the `coverage_stats` dictionary we populated in Step 7.
    
    df = coverage_stats[resolution].copy()
    
    # To make it renderable, we might need to simplify or just plot centroids if it's too dense.
    # But let's try plotting geometry. If Res 12 is too slow, we warn.
    
    # For minimaps, we want to see the PATTERN.
    # If we plot 170k 'covered' cells, it might be dense.
    
    # Optimization: Only calculate geometry for 'has_images' = True to save time?
    # User said "ensure ALL cells are included", so we must plot both.
    
    # Let's use a helper to generate geometry only for the plot
    # And maybe simplify the output.
    
    print(f"Generating geometry for Res {resolution} ({len(df)} cells)...")
    
    # If > 100k, we might want to just plot points? Or larger parent cells?
    # Let's try plotting parent cells at Res 10 for all of them? 
    # No, that defeats the purpose of showing different resolutions.
    
    # Let's just do it. It might take a minute.
    # Using vectorized h3.cells_to_h3shape (if available) or loop.
    
    # Faster geometry generation
    def fast_geom(h3_idx):
        return shape({'type': 'Polygon', 'coordinates': [[(p[1], p[0]) for p in h3.cell_to_boundary(h3_idx)]]})
    
    # For Res 12, this is slow. 
    # Let's strictly limit to a bounding box or just accept the slowness for the final output.
    # OR, we can use the 'vis_res' trick again just for the visual, but labeled as Res X.
    # BUT, the user asked for "each validated h3 coverage resolution".
    
    # Let's try a compromise: Plot the actual cells but use a low line width.
    
    df['geometry'] = df['h3_cell'].apply(fast_geom)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
    
    gdf.plot(column='image_count', ax=ax, 
             cmap='viridis', scheme='quantiles', k=5, 
             linewidth=0.0, # No borders for speed and clarity at high density
             legend=False)
    ax.set_title(f"Resolution {resolution}")
    ax.axis('off')

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5)) # Half height of the big one roughly

for i, res in enumerate([10, 11, 12]):
    plot_minimap(axes[i], coverage_stats, res)

plt.tight_layout()
plt.show()

## 9. Census Block & Block Group Analysis

Replicate the coverage analysis for Census Blocks (CB) and Census Block Groups (CBG) using the provided GeoJSON files.

In [ ]:
def analyze_census_coverage(geojson_path, dashcam_df, name="Census Unit", network_gdf=None):
    print(f"\n--- Analyzing Coverage for {name} ---")
    
    # 1. Load Census Geometries
    print(f"Loading {name} from {geojson_path}...")
    gdf_census = gpd.read_file(geojson_path)
    print(f"Loaded {len(gdf_census)} features.")
    
    # 2. Determine 'Coverage' for Census Units
    # A census unit is 'covered' if it contains AT LEAST ONE dashcam image.
    # To do this efficiently:
    # Method A: Spatial Join (Point in Polygon)
    #   - Create GeoDataFrame of dashcam points
    #   - sjoin with census polygons
    #   - count images per census ID
    
    # Convert dashcam data to GeoDataFrame
    # Note: We already have df_md with lat/lon. Let's make a GDF.
    # Doing this for 900k points might be heavy but standard for spatial analysis.
    
    print("Creating Dashcam GeoDataFrame...")
    gdf_dashcam = gpd.GeoDataFrame(
        dashcam_df, 
        geometry=gpd.points_from_xy(dashcam_df['gps_info.longitude'], dashcam_df['gps_info.latitude']),
        crs="EPSG:4326"
    )
    
    # Ensure census data is in same CRS
    if gdf_census.crs != gdf_dashcam.crs:
        gdf_census = gdf_census.to_crs(gdf_dashcam.crs)
        
    # Filter by intersecting with street network if provided
    if network_gdf is not None:
        print("Filtering census units that intersect with buffered street network...")
        initial_count = len(gdf_census)
        
        # Ensure CRS match
        if network_gdf.crs != gdf_census.crs:
            network_gdf_local = network_gdf.to_crs(gdf_census.crs)
        else:
            network_gdf_local = network_gdf
            
        # Perform spatial join (inner) to keep only intersecting rows
        # We only need the geometry from network to save memory/time
        # We use sjoin with predicate='intersects'
        gdf_census = gpd.sjoin(gdf_census, network_gdf_local[['geometry']], how='inner', predicate='intersects')
        
        # Remove duplicates
        gdf_census = gdf_census[~gdf_census.index.duplicated(keep='first')]
        
        # Clean up sjoin columns
        if 'index_right' in gdf_census.columns:
            gdf_census = gdf_census.drop(columns=['index_right'])
            
        print(f"Filtered from {initial_count} to {len(gdf_census)} features.")
    
    print("Performing Spatial Join...")
    # sjoin: for each image, find which census polygon it falls into
    # op='within' is faster than 'intersects' for points
    joined = gpd.sjoin(gdf_dashcam, gdf_census, how="inner", predicate="within")
    
    # 3. Aggregate Counts
    # We assume the census file has a unique identifier. Usually 'GEOID' or 'BoroCT2020' etc.
    # Let's inspect columns if generic, but we'll guess or find the first unique ID.
    # Looking at standard NYC files, it's likely 'GEOID' or similar.
    # Let's try to find a likely ID column.
    possible_ids = ['GEOID', 'geoid', 'BoroCT2020', 'cb2020', 'boroct2020', 'gid', 'id']
    id_col = next((col for col in possible_ids if col in gdf_census.columns), gdf_census.columns[0])
    print(f"Using ID column: {id_col}")
    
    # Count images per ID
    counts = joined[id_col].value_counts()
    
    # Merge back to full census gdf to include those with 0 images
    gdf_coverage = gdf_census.merge(
        counts.rename('image_count'),
        left_on=id_col,
        right_index=True,
        how='left'
    )
    
    gdf_coverage['image_count'] = gdf_coverage['image_count'].fillna(0).astype(int)
    gdf_coverage['has_images'] = gdf_coverage['image_count'] > 0
    
    # 4. Statistics
    total = len(gdf_coverage)
    covered = gdf_coverage['has_images'].sum()
    not_covered = total - covered
    pct_covered = (covered / total) * 100
    
    print(f"Total {name}s: {total}")
    print(f"{name}s with Images: {covered} ({pct_covered:.2f}%)")
    print(f"{name}s without Images: {not_covered} ({100 - pct_covered:.2f}%)")
    
    return gdf_coverage, id_col



# Paths provided by user
CBG_PATH = "../../data/cbg-nyc-2020.geojson"
CB_PATH = "../../data/cb-nyc-2020.geojson"
CT_PATH = "../../aggregation/geo/data/ct-nyc-2020.geojson"

# Run Analysis
gdf_ct_cov, ct_id = analyze_census_coverage(CT_PATH, df_md, "Census Tract", gdf_edges_buffered)
gdf_cbg_cov, cbg_id = analyze_census_coverage(CBG_PATH, df_md, "Census Block Group", gdf_edges_buffered)
gdf_cb_cov, cb_id = analyze_census_coverage(CB_PATH, df_md, "Census Block", gdf_edges_buffered)


## 10. Visualization of Census Coverage

Visualize the coverage maps for Census Block Groups and Census Blocks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Plot CBG
gdf_cbg_cov.plot(column='image_count', ax=axes[0], 
                 cmap='viridis', scheme='quantiles', k=5, 
                 legend=True, legend_kwds={'loc': 'lower right', 'title': 'Image Count'})
# fill the census block groups that have 0 images in red
gdf_cbg_cov.loc[gdf_cbg_cov['image_count'] == 0, 'no_images'] = 1
gdf_cbg_cov[gdf_cbg_cov['no_images'] == 1].plot(ax=axes[0], color='red', alpha=1)


axes[0].set_title("Census Block Groups Coverage")
axes[0].axis('off')

# Plot CB
gdf_cb_cov.plot(column='image_count', ax=axes[1], 
                cmap='viridis', scheme='quantiles', k=5, 
                legend=True, legend_kwds={'loc': 'lower right', 'title': 'Image Count'})
# fill the census blocks that have 0 images in red
gdf_cb_cov.loc[gdf_cb_cov['image_count'] == 0, 'no_images'] = 1
gdf_cb_cov[gdf_cb_cov['no_images'] == 1].plot(ax=axes[1], color='red', alpha=1)

axes[1].set_title("Census Blocks Coverage")
axes[1].axis('off')

# add text box to the top center of the plot 
text_box = plt.text(1.15, 0.95, 'No Images', ha='center', va='center', fontsize=12, color='red', transform=axes[0].transAxes)
plt.show()

In [ ]:
# for each geometry type, 